In [ ]:
import sysfrom pathlib import Path# Handle multiple working directory scenarios# Case 1: Running from project root# Case 2: Running from notebooks/ directory# Case 3: Running as a Jupyter notebookcurrent_file = Path(__file__).resolve() if '__file__' in dir() else Path.cwd()if current_file.is_file():    # We're running a notebook file    notebook_dir = current_file.parent    project_root = notebook_dir.parentelse:    # We're in a directory    project_root = Path.cwd()    # If we're in notebooks/ subdirectory, go up one level    if project_root.name == 'notebooks':        project_root = project_root.parent# Add project root to path if not already thereif str(project_root) not in sys.path:    sys.path.insert(0, str(project_root))# Verify src module existssrc_path = project_root / 'src'if not src_path.exists():    raise RuntimeError(f"Could not find src/ directory. Project root: {project_root}")print(f"✓ Added {project_root} to sys.path")

# Training and Generating with TinyMaskedLM

This notebook demonstrates iterative denoising generation with masked language models.

We'll:
1. Train masked LM on a toy corpus
2. Understand the masked token prediction objective
3. Generate text through iterative unmasking
4. Compare generation with different step counts

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from src.diffusion.masked_lm import TinyMaskedLM
from src.common.tokenizer import CharTokenizer

torch.manual_seed(42)
print("✓ Imports successful")

## Part 1: Setup

Key point: Token ID 0 is reserved for MASK tokens in masked LM.

In [ ]:
corpus = "the quick brown fox jumps over the lazy dog"
tokenizer = CharTokenizer(corpus)

print(f"Corpus: '{corpus}'")
print(f"Tokenizer vocab size: {tokenizer.vocab_size}")
print(f"Masked LM vocab size: {tokenizer.vocab_size + 1} (includes MASK)")
print(f"\nMask token ID: 0 (reserved)")
print(f"Token IDs: 1 to {tokenizer.vocab_size}")

# Tokenize and shift by 1 (ID 0 is for MASK)
token_ids = torch.tensor([tokenizer.encode(corpus)]) + 1
print(f"\nTokenized and shifted: {token_ids[0][:10].tolist()} ...")

## Part 2: Create and Train Masked LM

In [ ]:
masked_lm = TinyMaskedLM(
    vocab_size=tokenizer.vocab_size + 1,  # +1 for MASK
    d_model=64,
    n_layers=2,
    n_heads=4
)

print(f"Model created")
print(f"Total parameters: {sum(p.numel() for p in masked_lm.parameters()):,}")

optimizer = torch.optim.Adam(masked_lm.parameters(), lr=0.01)
num_epochs = 100
losses = []

print(f"\nTraining for {num_epochs} epochs...")
print(f"{'Epoch':<8} {'Loss':<12}")
print("-" * 20)

for epoch in range(num_epochs):
    optimizer.zero_grad()
    
    # Create random mask (50% of tokens)
    masked_input = token_ids.clone()
    mask = torch.rand_like(token_ids, dtype=torch.float) < 0.5
    masked_input[mask] = 0  # Replace with MASK token
    
    # Forward pass
    logits = masked_lm(masked_input)
    
    # Loss: predict original tokens at masked positions only
    if mask.any():
        loss = F.cross_entropy(
            logits[mask].reshape(-1, masked_lm.vocab_size),
            token_ids[mask].reshape(-1)
        )
        loss.backward()
        optimizer.step()
    else:
        loss = torch.tensor(0.0)
    
    losses.append(loss.item())
    
    if epoch % 20 == 0:
        print(f"{epoch:<8} {loss.item():<12.4f}")

print(f"{num_epochs:<8} {losses[-1]:<12.4f}")
print(f"✓ Training complete")

## Part 3: Visualize Training Curve

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(losses, linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Masked LM Training Loss (Full)')
ax1.grid(True, alpha=0.3)

ax2.plot(losses[-20:], linewidth=2, color='orange')
ax2.set_xlabel('Epoch (last 20)')
ax2.set_ylabel('Loss')
ax2.set_title('Training Loss (Final Phase)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Starting loss: {losses[0]:.4f}")
print(f"Final loss: {losses[-1]:.4f}")

## Part 4: Generation through Iterative Unmasking

In [ ]:
print("="*70)
print("ITERATIVE UNMASKING GENERATION")
print("="*70)
print()
print("Algorithm:")
print("1. Start: all tokens = MASK (0)")
print("2. For each step:")
print("   - Forward pass to get predictions")
print("   - Unmask ~1/steps most confident positions")
print("   - Repeat until all unmasked")
print()

target_length = 15

for steps in [2, 4, 8]:
    with torch.no_grad():
        generated = masked_lm.generate(length=target_length, steps=steps)
    
    # Shift back: ID 0 is MASK, 1+ are tokens
    tokens = [t.item() - 1 for t in generated[0] if t > 0]
    text = tokenizer.decode(tokens)
    
    print(f"Steps={steps:<2}: '{text}'")

## Part 5: Understand Iterative Refinement

In [ ]:
print("\n" + "="*70)
print("WHY ITERATIVE UNMASKING WORKS")
print("="*70)
print()
print("Bidirectional Context:")
print("  - At each step, ALL unmasked tokens inform predictions")
print("  - No left/right bias, can attend in all directions")
print()
print("Refinement over steps:")
print("  Step 1: Start all masked, broad predictions")
print("  Step 2: Some tokens fixed, predictions for rest refine")
print("  Step 3: More tokens fixed, even better context")
print("  ...")
print("  Step N: All tokens unmasked, final sequence")
print()
print("Key Difference from GPT:")
print("  GPT:       Sequential generation (left-to-right)")
print("  Masked LM: Iterative refinement (all positions at once)")
print()
print("Trade-offs:")
print("  GPT:       Faster (N tokens = N forward passes)")
print("  Masked LM: Slower (N tokens = steps*N forward passes)")
print("             But: can revise earlier positions as context builds")

## Summary

**Masked Language Model for Generation:**

✓ **Training Objective:** Predict masked tokens given full context  
✓ **Bidirectional Attention:** All positions see all context  
✓ **Generation:** Iterative unmasking with refinement  
✓ **Multiple Steps:** More steps = more refinement iterations  

**Comparison with GPT:**

| Aspect | GPT | Masked LM |
|--------|-----|----------|
| Attention | Causal (left only) | Bidirectional |
| Training | Next-token | Denoising |
| Generation | Sequential | Iterative |
| Context | Left only | Full |
| Can revise | No | Yes |
| Speed | O(N) | O(steps*N) |

**Same Foundation:**  
Both use identical MultiHeadSelfAttention block.  
Different `causal` flag enables these fundamentally different approaches.